In [2]:
import pandas as pd
from datetime import datetime, timedelta
import requests
from pytz import timezone



# Making the API call and viewing the json

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

icao = "EDDB"
date = datetime.now().date()
time_1 = "00:00"
time_2 = "11:59"

url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{date}T{time_1}/{date}T{time_2}"

querystring = {"withLeg":"true",
               "direction":"Arrival",
               "withCancelled":"false",
               "withCodeshared":"true",
               "withCargo":"false",
               "withPrivate":"false"}

headers = {
    'x-rapidapi-host': "aerodatabox.p.rapidapi.com",
    "x-rapidapi-key": os.getenv("x_rapidapi_key"),
    }

response = requests.request("GET",
                            url,
                            headers = headers,
                            params = querystring)

flights_json = response.json()

flights_json

{'arrivals': [{'departure': {'airport': {'icao': 'ZBAA',
     'iata': 'PEK',
     'name': 'Beijing',
     'countryCode': 'cn',
     'timeZone': 'Asia/Shanghai'},
    'scheduledTime': {'utc': '2026-05-28 19:10Z',
     'local': '2026-05-29 03:10+08:00'},
    'revisedTime': {'utc': '2026-05-28 19:46Z',
     'local': '2026-05-29 03:46+08:00'},
    'terminal': '2',
    'checkInDesk': 'D01-D20',
    'gate': '08',
    'quality': ['Basic', 'Live']},
   'arrival': {'scheduledTime': {'utc': '2026-05-29 04:45Z',
     'local': '2026-05-29 06:45+02:00'},
    'revisedTime': {'utc': '2026-05-29 04:38Z',
     'local': '2026-05-29 06:38+02:00'},
    'terminal': '1',
    'gate': 'Y17',
    'baggageBelt': 'B3',
    'quality': ['Basic', 'Live']},
   'number': 'HU 489',
   'callSign': 'CHH489',
   'status': 'Arrived',
   'codeshareStatus': 'IsOperator',
   'isCargo': False,
   'aircraft': {'reg': 'B-1135', 'modeS': '78140C', 'model': 'Boeing 787-9'},
   'airline': {'name': 'Hainan', 'iata': 'HU', 'icao': '

# Exploring the json

In [4]:
flights_json.keys()

dict_keys(['arrivals'])

In [5]:
flights_json["arrivals"]

[{'departure': {'airport': {'icao': 'ZBAA',
    'iata': 'PEK',
    'name': 'Beijing',
    'countryCode': 'cn',
    'timeZone': 'Asia/Shanghai'},
   'scheduledTime': {'utc': '2026-05-28 19:10Z',
    'local': '2026-05-29 03:10+08:00'},
   'revisedTime': {'utc': '2026-05-28 19:46Z',
    'local': '2026-05-29 03:46+08:00'},
   'terminal': '2',
   'checkInDesk': 'D01-D20',
   'gate': '08',
   'quality': ['Basic', 'Live']},
  'arrival': {'scheduledTime': {'utc': '2026-05-29 04:45Z',
    'local': '2026-05-29 06:45+02:00'},
   'revisedTime': {'utc': '2026-05-29 04:38Z',
    'local': '2026-05-29 06:38+02:00'},
   'terminal': '1',
   'gate': 'Y17',
   'baggageBelt': 'B3',
   'quality': ['Basic', 'Live']},
  'number': 'HU 489',
  'callSign': 'CHH489',
  'status': 'Arrived',
  'codeshareStatus': 'IsOperator',
  'isCargo': False,
  'aircraft': {'reg': 'B-1135', 'modeS': '78140C', 'model': 'Boeing 787-9'},
  'airline': {'name': 'Hainan', 'iata': 'HU', 'icao': 'CHH'}},
 {'departure': {'airport': {'ica

The square brackets at the beginning of `flights_json` indicate that it represents a list-like structure. Since we're dealing with a list, we can iterate through its elements to access and process the data. Let's start by examining the first element in the list.

In [6]:
flights_json["arrivals"][0]

{'departure': {'airport': {'icao': 'ZBAA',
   'iata': 'PEK',
   'name': 'Beijing',
   'countryCode': 'cn',
   'timeZone': 'Asia/Shanghai'},
  'scheduledTime': {'utc': '2026-05-28 19:10Z',
   'local': '2026-05-29 03:10+08:00'},
  'revisedTime': {'utc': '2026-05-28 19:46Z',
   'local': '2026-05-29 03:46+08:00'},
  'terminal': '2',
  'checkInDesk': 'D01-D20',
  'gate': '08',
  'quality': ['Basic', 'Live']},
 'arrival': {'scheduledTime': {'utc': '2026-05-29 04:45Z',
   'local': '2026-05-29 06:45+02:00'},
  'revisedTime': {'utc': '2026-05-29 04:38Z',
   'local': '2026-05-29 06:38+02:00'},
  'terminal': '1',
  'gate': 'Y17',
  'baggageBelt': 'B3',
  'quality': ['Basic', 'Live']},
 'number': 'HU 489',
 'callSign': 'CHH489',
 'status': 'Arrived',
 'codeshareStatus': 'IsOperator',
 'isCargo': False,
 'aircraft': {'reg': 'B-1135', 'modeS': '78140C', 'model': 'Boeing 787-9'},
 'airline': {'name': 'Hainan', 'iata': 'HU', 'icao': 'CHH'}}

In [7]:
flights_json["arrivals"][0].keys()

dict_keys(['departure', 'arrival', 'number', 'callSign', 'status', 'codeshareStatus', 'isCargo', 'aircraft', 'airline'])

Looking at the first element of the json and the available keys, we can select the information we think would be important for our dataframe.
- Departure airport icao
- scheduled arrival time, local
- flight number

# Using for loops

## Making the DataFrame

In [8]:
flight_items = []

for item in flights_json["arrivals"]:
  flight_item = {
      "arrival_airport_icao": icao,
      "departure_airport_icao": item["departure"]["airport"].get("icao", None),
      "scheduled_arrival_time": item["arrival"]["scheduledTime"].get("local", None),
      "flight_number": item.get("number", None)
  }

  flight_items.append(flight_item)

flights_df = pd.DataFrame(flight_items)

flights_df.head()

,arrival_airport_icao,departure_airport_icao,scheduled_arrival_time,flight_number
0,EDDB,ZBAA,2026-05-29 06:45+02:00,HU 489
1,EDDB,KEWR,2026-05-29 07:15+02:00,UA 962
2,EDDB,LROP,2026-05-29 07:05+02:00,W4 3109
3,EDDB,LTAJ,2026-05-29 06:45+02:00,XQ 1766
4,EDDB,LZIB,2026-05-29 07:15+02:00,W6 7037


Let's get rid of the `+01:00` from `scheduled_arrival_time`.

In [9]:
flights_df["scheduled_arrival_time"] = flights_df["scheduled_arrival_time"].str[:-6]
flights_df.head()

,arrival_airport_icao,departure_airport_icao,scheduled_arrival_time,flight_number
0,EDDB,ZBAA,2026-05-29 06:45,HU 489
1,EDDB,KEWR,2026-05-29 07:15,UA 962
2,EDDB,LROP,2026-05-29 07:05,W4 3109
3,EDDB,LTAJ,2026-05-29 06:45,XQ 1766
4,EDDB,LZIB,2026-05-29 07:15,W6 7037


While string slicing provides a quick solution to correcting the `scheduled_arrival_time` column, it's not the most robust approach. This is because it assumes that every cell in the column has the `+01:00` time zone offset. If there are cells without this offset, slicing would remove part of the time value, leading to inaccurate results.

A more robust solution would involve using the `re.sub()` function from the re module. Feel free to look into this if you have extra time and are curious.

## Creating a function for multiple cities

In [20]:

import pandas as pd
from datetime import datetime, timedelta
import requests
from pytz import timezone
from time import sleep

def get_flight_data(icao_list):
  
  berlin_timezone = timezone('Europe/Berlin')
  today = datetime.now(berlin_timezone).date()
  tomorrow = (today + timedelta(days=1))

  flight_items = []

  for icao in icao_list:
    # the api can only make 12 hour calls, therefore, two 12 hour calls make a full day
    # using the nested lists below we can make a morning call and extract the data
    # then make an afternoon call and extract the data
    times = [["00:00","11:59"],
             ["12:00","23:59"]]

    for time in times:
      url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{tomorrow}T{time[0]}/{tomorrow}T{time[1]}"

      querystring = {"withLeg":"true",
                    "direction":"Arrival",
                    "withCancelled":"false",
                    "withCodeshared":"false",
                    "withCargo":"false",
                    "withPrivate":"false"}

      headers = {
          'x-rapidapi-host': "aerodatabox.p.rapidapi.com",
          'x-rapidapi-key': os.getenv("x_rapidapi_key")
          }
      sleep(2)
      response = requests.request("GET",
                                  url,
                                  headers = headers,
                                  params = querystring)

      flights_json = response.json()

      retrieval_time = datetime.now(berlin_timezone).strftime("%Y-%m-%d %H:%M:%S")

      for item in flights_json["arrivals"]:
        flight_item = {
            "arrival_airport_icao": icao,
            "departure_airport_icao": item["departure"]["airport"].get("icao", None),
            "scheduled_arrival_time": item["arrival"]["scheduledTime"].get("local", None),
            "flight_number": item.get("number", None),
            "data_retrieved_at": retrieval_time
        }

        flight_items.append(flight_item)

  flights_df = pd.DataFrame(flight_items)
  flights_df["scheduled_arrival_time"] = flights_df["scheduled_arrival_time"].str[:-6]
  flights_df["scheduled_arrival_time"] = pd.to_datetime(flights_df["scheduled_arrival_time"])
  flights_df["data_retrieved_at"] = pd.to_datetime(flights_df["data_retrieved_at"])

  return flights_df

In [21]:
icao_list = ["EDDB", "EDDF"]

get_flight_data(icao_list)

,arrival_airport_icao,departure_airport_icao,scheduled_arrival_time,flight_number,data_retrieved_at
0,EDDB,LTBJ,2026-05-30 06:05:00,XQ 966,2026-05-29 16:07:53
1,EDDB,LTDB,2026-05-30 06:45:00,XQ 1774,2026-05-29 16:07:53
2,EDDB,LROP,2026-05-30 07:00:00,W4 3109,2026-05-29 16:07:53
3,EDDB,KEWR,2026-05-30 07:15:00,UA 962,2026-05-29 16:07:53
4,EDDB,LIME,2026-05-30 07:25:00,FR 2669,2026-05-29 16:07:53
...,...,...,...,...,...
868,EDDF,GCRR,2026-05-30 22:40:00,4Y 309,2026-05-29 16:08:00
869,EDDF,LPMA,2026-05-30 22:40:00,DE 1571,2026-05-29 16:08:00
870,EDDF,GCLP,2026-05-30 22:40:00,4Y 303,2026-05-29 16:08:00
871,EDDF,HEGN,2026-05-30 22:45:00,DE 31,2026-05-29 16:08:00


Your flight function can now be incorporated with your other functions to send and receive data from your SQL database.

# Using json_normalize

In [18]:
from time import sleep
def tomorrows_flight_arrivals(icao_list):

    

    berlin_timezone = timezone('Europe/Berlin')
    today = datetime.now(berlin_timezone).date()
    tomorrow = (today + timedelta(days=1))

    list_for_arrivals_df = []

    for icao in icao_list:

        times = [["00:00","11:59"],["12:00","23:59"]]

        for time in times:
            url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{tomorrow}T{time[0]}/{tomorrow}T{time[1]}"

            querystring = {"direction":"Arrival","withCancelled":"false"}

            headers = {
                "x-rapidapi-key": os.getenv("x_rapidapi_key"),
                "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
                }
            sleep(2)
            response = requests.request("GET", url, headers=headers, params=querystring)
            flights_resp = response.json()

            arrivals_df = pd.json_normalize(flights_resp["arrivals"])[["number", "airline.name", "movement.scheduledTime.local", "movement.terminal", "movement.airport.name", "movement.airport.icao"]]
            arrivals_df = arrivals_df.rename(columns={"number": "flight_number", "airline.name": "airline", "movement.scheduledTime.local": "arrival_time", "movement.terminal": "arrival_terminal", "movement.airport.name": "departure_city", "movement.airport.icao": "departure_airport_icao"})
            arrivals_df["arrival_airport_icao"] = icao
            arrivals_df["data_retrieved_on"] = datetime.now(berlin_timezone).strftime("%Y-%m-%d %H:%M:%S")
            arrivals_df = arrivals_df[["arrival_airport_icao", "flight_number", "airline", "arrival_time", "arrival_terminal", "departure_city", "departure_airport_icao", "data_retrieved_on"]]

            # fixing arrival_time
            arrivals_df["arrival_time"] = arrivals_df["arrival_time"].str.split("+").str[0]

            list_for_arrivals_df.append(arrivals_df)

    return pd.concat(list_for_arrivals_df, ignore_index=True)

In [19]:
icao_list = ["EDDF", "EDDB"]

tomorrows_flight_arrivals(icao_list)

,arrival_airport_icao,flight_number,airline,arrival_time,arrival_terminal,departure_city,departure_airport_icao,data_retrieved_on
0,EDDF,3S 617,AeroLogic,2026-05-30 04:58,NaN,Los Angeles,KLAX,2026-05-29 15:53:32
1,EDDF,LH 1327,Lufthansa,2026-05-30 05:20,1,Tunis,DTTA,2026-05-29 15:53:32
2,EDDF,AC 9277,Air Canada,2026-05-30 05:20,1,Tunis,DTTA,2026-05-29 15:53:32
3,EDDF,LH 1319,Lufthansa,2026-05-30 05:20,1,Algiers,DAAG,2026-05-29 15:53:32
4,EDDF,AC 9359,Air Canada,2026-05-30 05:20,1,Algiers,DAAG,2026-05-29 15:53:32
...,...,...,...,...,...,...,...,...
2832,EDDB,FR 2733,Ryanair,2026-05-30 22:55,2,Tallinn,EETN,2026-05-29 15:53:38
2833,EDDB,XQ 4143,Sun Express,2026-05-30 23:00,1,Málaga,LEMG,2026-05-29 15:53:38
2834,EDDB,EW 8537,Eurowings,2026-05-30 23:00,1,Málaga,LEMG,2026-05-29 15:53:38
2835,EDDB,FR 5418,Ryanair,2026-05-30 23:00,2,Dublin,EIDW,2026-05-29 15:53:38
